# vgc-bot — end to end on Colab Pro (A100/L4)

Pipeline: **setup → data → train → evaluate → env benchmark → search**.

Each section works independently given the previous section's artifacts on Drive.

## 0. Setup — repo, Python deps, Node + pinned sim

In [ ]:
%cd /content
!git clone https://github.com/YOURNAME/vgc-bot.git || true
%cd /content/vgc-bot
!pip install -q -r requirements.txt

In [ ]:
# Node 22 + pinned pokemon-showdown (git commit with Champions formats) + @smogon/calc.
# The git install ships no dist/, so build it in place (~2 min).
!node --version || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -q nodejs)
!mkdir -p artifacts/node && cp -n /dev/null artifacts/node/.keep
%cd /content/vgc-bot/artifacts/node
![ -f package.json ] || echo '{"name":"vgc-bot-node","private":true}' > package.json
!npm install --no-audit --no-fund github:smogon/pokemon-showdown#e440c4a18385274f10c405d0b158b6a962ce6d94 @smogon/calc@0.11.0
!cd node_modules/pokemon-showdown && (npm install --no-audit --no-fund || true) && node build && ls dist/sim/index.js
%cd /content/vgc-bot

In [ ]:
# Checkpoints go to Drive so training is resumable across sessions.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/vgc-bot/checkpoints
!sed -i 's|Path("artifacts/checkpoints")|Path("/content/drive/MyDrive/vgc-bot/checkpoints")|' config.py
!grep checkpoint_dir config.py

## 1. Data — download, parse, prep (beliefs + damage features + tokenize)

In [ ]:
!python env.py --dump-dex   # champions dex for stat/priority math
!python data.py download    # ~630 MB from Hugging Face
!python data.py parse

In [ ]:
# The slow step (particle filter + damage calc per transition). Cache artifacts
# to Drive afterwards if you want to skip it next session.
!python data.py prep

## 2. Train — behavior cloning (resumes from ckpt_last.pt automatically)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/vgc-bot/checkpoints/tb
!python train.py

## 3. Evaluate — predictor benchmarks on held-out battles

In [ ]:
!python evaluate.py

## 4. Env benchmark — sidecar save/restore proof + steps/sec (sets the search budget)

In [ ]:
!python env.py --benchmark 3000

## 5. Phase 2 — search
Reconstruction selftest first (proves the sim pin), then the belief-filter audit,
the scenario assertions (incl. the Metagross/Kingambit mixed-strategy gate) and a
watchable search-vs-search game.

NOTE: if the prepped shards/checkpoints on Drive predate the phase-2 tokenizer
layout (belief tokens + damage ranges), re-run sections 1–2 first.

In [ ]:
!python env.py --selftest      # mid-battle reconstruction proof
!python beliefs.py --audit 300 # particle-filter breakdown on held-out games

In [ ]:
!python scenarios.py           # endgame assertions; works pre-training too
!python scenarios.py --mine    # real-replay endgame candidates -> artifacts/

In [ ]:
!python observe_game.py --games 1 --temp 0.5   # needs ckpt_best.pt

## Phase 3 (next)
Bot-vs-bot round-robin Elo: random / max-damage / policy-only / policy+search.
The policy-only vs policy+search gap is the evidence that search adds value.